# AeroPrice: hybrid model retraining on Kaggle

Trains and compares two hybrid AeroPrice models on the same split, using the **satellite image + numeric property features** (living area, year built, bedrooms, bathrooms, school rating, lot size):

1. **Neural hybrid:** a ResNet50 image branch and a dense numeric branch, fused and trained end to end.
2. **Deep + boosting hybrid:** ResNet50 image features, compressed with PCA, plus the numeric features, fed to gradient boosting.

Both are compared with a numbers-only model and a guess-the-median baseline in one table.

**Before you run**
1. *Settings → Accelerator →* **GPU T4 x2** (or P100). Training on CPU will take many hours.
2. *Settings →* **Internet on** (needed once to download the ImageNet weights for ResNet50; Kaggle may ask you to verify your phone number first).
3. *Add Input →* the dataset **sanveersinghsuneja/austin-satellite-property-data**.
4. *Run All*. Takes roughly 25–70 minutes on a GPU.

**Afterwards:** download **`aeroprice_trained_model.zip`** from the Output panel (right side, `/kaggle/working`) and import it on your laptop with:

```
python pipeline/import_kaggle_model.py path\to\aeroprice_trained_model.zip
```

Weights are exported layer-by-layer by name, so they load into TensorFlow 2.10 on your laptop whatever Keras version Kaggle uses.

In [ ]:
import os, glob, json, time, zipfile
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from sklearn.model_selection import train_test_split

layers = keras.layers
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU") or "none (this will be slow)")

## Settings

In [ ]:
QUICK_TEST = False        # True = tiny run to check the notebook works end to end
USE_LATLON = False        # True = also feed latitude/longitude (usually a big accuracy gain)

NUMERIC_FEATURES = ["livingAreaSqFt", "yearBuilt", "numOfBedrooms",
                    "numOfBathrooms", "avgSchoolRating", "lotSizeSqFt"]
if USE_LATLON:
    NUMERIC_FEATURES += ["latitude", "longitude"]

IMAGE_SIZE = 128
BATCH_SIZE = 32
INITIAL_EPOCHS = 15       # stage 1: ResNet50 frozen
FINE_TUNE_EPOCHS = 35     # stage 2: last FINE_TUNE_LAYERS of ResNet50 unfrozen
FINE_TUNE_LAYERS = 20
LR_INITIAL = 1e-3
LR_FINE_TUNE = 1e-5
SEED = 42
PRETRAINED = True        # ImageNet weights for ResNet50 (needs Internet on)

# Model 2 (deep + boosting hybrid)
N_IMG_COMPONENTS = 32    # image features kept after PCA
GB_PARAMS = dict(max_iter=1000, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,
                 l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
                 n_iter_no_change=30, random_state=0)

DATA_ROOTS = ["/kaggle/input"]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "kaggle_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

if QUICK_TEST:
    INITIAL_EPOCHS, FINE_TUNE_EPOCHS = 1, 1
tf.keras.utils.set_random_seed(SEED)

## Find the data

In [ ]:
def find_one(pattern):
    hits = []
    for root in DATA_ROOTS:
        hits += glob.glob(os.path.join(root, "**", pattern), recursive=True)
    return sorted(hits, key=len)

csv_hits = find_one("austin_master_dataset.csv")
img_hits = find_one("property_0_rgb.png")
assert csv_hits, "austin_master_dataset.csv not found. Did you add the dataset with 'Add Input'?"
assert img_hits, "Image patches (property_*_rgb.png) not found in the attached dataset."
CSV_PATH = csv_hits[0]
IMAGE_DIR = os.path.dirname(img_hits[0])
print("CSV:   ", CSV_PATH)
print("Images:", IMAGE_DIR)

## Load, clean and split
The 80/20 split uses `random_state=42` exactly like the original project, so `pipeline/evaluate_model.py` on your laptop scores the same 20% test set.

In [ ]:
df = pd.read_csv(CSV_PATH)
df["image_file"] = df["rgb_path"].astype(str).str.replace("\\", "/", regex=False).map(os.path.basename)
available = set(os.listdir(IMAGE_DIR))
df = df[df["rgb_path"].astype(str).str.len() > 0]
df = df[df["image_file"].isin(available)].copy()
df["latestPrice"] = pd.to_numeric(df["latestPrice"], errors="coerce")
df = df.dropna(subset=["latestPrice"]).reset_index(drop=True)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["image_path"] = IMAGE_DIR + "/" + df["image_file"]
print(f"{len(df)} properties with images")

trainval_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.1, random_state=42)
if QUICK_TEST:
    train_idx, val_idx, test_idx = train_idx[:256], val_idx[:64], test_idx[:64]
print(f"train={len(train_idx)}  val (early stopping)={len(val_idx)}  test (untouched)={len(test_idx)}")

# Numeric features: fill gaps with the training median, standardise with training stats
medians = df.loc[train_idx, NUMERIC_FEATURES].median()
X_all = df[NUMERIC_FEATURES].fillna(medians).to_numpy("float32")
mean = X_all[train_idx].mean(axis=0)
scale = X_all[train_idx].std(axis=0)
scale[scale == 0] = 1.0
X_all = (X_all - mean) / scale
y_all = df["latestPrice"].to_numpy("float32")
paths_all = df["image_path"].to_numpy()
print(pd.DataFrame({"median": medians, "mean": mean, "std": scale}))

## Input pipeline

In [ ]:
# ResNet50 'caffe' preprocessing (same maths as keras.applications.resnet50.preprocess_input)
MEAN_BGR = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)

def load_image(path):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return tf.clip_by_value(img, 0.0, 255.0)

def caffe_preprocess(img):
    return img[..., ::-1] - MEAN_BGR

def augment(img):
    img = img / 255.0
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.image.random_saturation(img, 0.9, 1.1)
    return tf.clip_by_value(img, 0.0, 1.0) * 255.0

def make_dataset(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((paths_all[idx], X_all[idx], np.log1p(y_all[idx])))
    ds = ds.map(lambda p, t, y: (load_image(p), t, y), num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(idx), seed=SEED).map(lambda i, t, y: (augment(i), t, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda i, t, y: ({"image_input": caffe_preprocess(i), "tabular_input": t}, y))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_idx, training=True)
val_ds = make_dataset(val_idx, training=False)
test_ds = make_dataset(test_idx, training=False)

## Model 1: neural hybrid
Same architecture as `pipeline/model_def.py` in the project (keep the two identical).

In [ ]:
def build_multimodal_model(num_tab_features=0, image_size=128, pretrained=True):
    image_input = layers.Input(shape=(image_size, image_size, 3), name="image_input")
    resnet = keras.applications.ResNet50(include_top=False, weights="imagenet" if pretrained else None,
                                         input_shape=(image_size, image_size, 3))
    resnet.trainable = False
    x_img = resnet(image_input, training=False)
    x_img = layers.Flatten(name="img_flatten")(x_img)
    x_img = layers.Dense(256, activation="relu", name="img_dense")(x_img)
    x_img = layers.Dropout(0.3, name="img_dropout")(x_img)
    x_img = layers.BatchNormalization(name="img_bn")(x_img)
    if num_tab_features > 0:
        tabular_input = layers.Input(shape=(num_tab_features,), name="tabular_input")
        x_num = layers.Dense(128, activation="relu", name="tab_dense")(tabular_input)
        x_num = layers.Dropout(0.2, name="tab_dropout")(x_num)
        x_num = layers.BatchNormalization(name="tab_bn")(x_num)
        fused = layers.Concatenate(name="fusion")([x_img, x_num])
        inputs = [image_input, tabular_input]
    else:
        fused = x_img
        inputs = image_input
    fused = layers.Dense(128, activation="relu", name="fused_dense")(fused)
    fused = layers.Dropout(0.3, name="fused_dropout")(fused)
    output = layers.Dense(1, activation="linear", name="price")(fused)
    model = keras.Model(inputs=inputs, outputs=output, name="satellite_price_model")
    return model

model = build_multimodal_model(len(NUMERIC_FEATURES), IMAGE_SIZE, pretrained=PRETRAINED)
# Start the output at the average log-price (about 13) instead of 0, so training does not waste epochs getting there
model.get_layer("price").bias.assign([float(np.log1p(y_all[train_idx]).mean())])
backbone = next(l for l in model.layers if isinstance(l, keras.Model))
model.summary()

## Train model 1
Stage 1 trains the new layers with ResNet50 frozen. Stage 2 fine-tunes the last ResNet50 layers at a low learning rate.

In [ ]:
t0 = time.time()
model.compile(optimizer=keras.optimizers.Adam(LR_INITIAL), loss="mse", metrics=["mae"])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])

backbone.trainable = True
for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(LR_FINE_TUNE), loss="mse", metrics=["mae"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)])
print(f"Training took {(time.time() - t0) / 60:.1f} min")

history = pd.concat([pd.DataFrame(h1.history).assign(stage=1), pd.DataFrame(h2.history).assign(stage=2)], ignore_index=True)
history.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index_label="epoch")
history[["loss", "val_loss"]].plot(title="log-price MSE", figsize=(7, 3));

## Evaluate model 1 on the untouched 20% test set
`price_scale` is a smearing correction measured on the validation set. Training on log(price) makes raw predictions systematically low in dollars, and this factor corrects that.

In [ ]:
def predict_prices(ds):
    return np.expm1(model.predict(ds, verbose=0).reshape(-1).astype("float64"))

val_pred = predict_prices(val_ds)
price_scale = float(np.mean(np.exp(np.log1p(y_all[val_idx]) - np.log1p(val_pred))))

def report(actual, pred, label):
    err = pred - actual
    pct = np.abs(err) / actual * 100
    la, lp = np.log1p(actual), np.log1p(pred)
    r2 = 1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)
    return {"model": label, "MAE $": round(np.abs(err).mean()), "median % error": round(np.median(pct), 1),
            "within 20 %": round((pct <= 20).mean() * 100, 1), "R2 (log)": round(r2, 3),
            "bias $": round(err.mean()), "max pred $": round(pred.max())}

y_test = y_all[test_idx].astype("float64")
nn_test_raw = predict_prices(test_ds)
nn_test_pred = nn_test_raw * price_scale
print(f"price_scale = {price_scale:.4f}")
pd.DataFrame([report(y_test, nn_test_raw, "1. neural hybrid (before price_scale)"),
              report(y_test, nn_test_pred, "1. neural hybrid")])

## Model 2: deep features + gradient boosting
ResNet50 with ImageNet weights (not fine-tuned) turns each satellite patch into 2,048 numbers. PCA, fitted on training rows only, compresses them to `N_IMG_COMPONENTS` image features. Gradient boosting then learns log(price) from those plus the raw numeric features. Tree models get exact, fast explanations (TreeSHAP).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

feature_extractor = keras.applications.ResNet50(include_top=False, pooling="avg", weights="imagenet" if PRETRAINED else None,
                                                input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
feat_rows = np.arange(len(df)) if not QUICK_TEST else np.concatenate([train_idx, val_idx, test_idx])
feat_ds = (tf.data.Dataset.from_tensor_slices(paths_all[feat_rows])
           .map(lambda p: caffe_preprocess(load_image(p)), num_parallel_calls=tf.data.AUTOTUNE)
           .batch(64).prefetch(tf.data.AUTOTUNE))
t0 = time.time()
deep = np.zeros((len(df), 2048), dtype="float32")
deep[feat_rows] = feature_extractor.predict(feat_ds, verbose=0)
has_features = np.zeros(len(df), dtype=bool)
has_features[feat_rows] = True
print(f"Image features for {len(feat_rows)} properties in {time.time() - t0:.0f}s")

# PCA by SVD on training rows only, stored as plain arrays so it works with any library version
pca_mean = deep[train_idx].mean(axis=0)
_, sv, vt = np.linalg.svd(deep[train_idx] - pca_mean, full_matrices=False)
pca_components = vt[:N_IMG_COMPONENTS].astype("float32")
img_pca = ((deep - pca_mean) @ pca_components.T).astype("float32")
explained = float((sv[:N_IMG_COMPONENTS] ** 2).sum() / (sv ** 2).sum())
print(f"{N_IMG_COMPONENTS} image components keep {explained * 100:.1f}% of the image-feature variance")

IMAGE_FEATURES = [f"img_pc{i + 1}" for i in range(N_IMG_COMPONENTS)]
X_raw = df[NUMERIC_FEATURES].fillna(medians).to_numpy("float32")   # trees need no scaling
X_hyb = np.hstack([X_raw, img_pca])

def fit_boosting(X):
    gbm = HistGradientBoostingRegressor(**GB_PARAMS).fit(X[train_idx], np.log1p(y_all[train_idx]))
    val = np.expm1(gbm.predict(X[val_idx]))
    scale_ = float(np.mean(np.exp(np.log1p(y_all[val_idx]) - np.log1p(val))))
    return gbm, scale_, np.expm1(gbm.predict(X[test_idx])) * scale_

gb_hybrid, gb_price_scale, gb_test_pred = fit_boosting(X_hyb)
gb_numbers, _, num_test_pred = fit_boosting(X_raw)
print(f"boosting price_scale = {gb_price_scale:.4f}, trees used = {gb_hybrid.n_iter_}")

## Compare all models on the same untouched test set

In [ ]:
results = pd.DataFrame([
    report(y_test, nn_test_pred, "1. neural hybrid (image + numbers)"),
    report(y_test, gb_test_pred, "2. deep + boosting hybrid (image features + numbers)"),
    report(y_test, num_test_pred, "numbers only, boosting (no image)"),
    {"model": "guess the median", "MAE $": round(np.abs(y_test - np.median(y_all[train_idx])).mean())},
])
recommended_model = "neural_hybrid" if results.loc[0, "median % error"] <= results.loc[1, "median % error"] else "boosting_hybrid"
print("Lower median % error wins. Recommended for the app:", recommended_model)
results

## What drives model 2? (global explanation)
Mean absolute SHAP value per feature on the test set, with the image components grouped as one *Satellite image* feature. Larger means the feature moves predictions more.

In [ ]:
try:
    import shap
    import matplotlib.pyplot as plt
    expl = shap.TreeExplainer(gb_hybrid)
    sv_test = expl.shap_values(X_hyb[test_idx])
    k = len(NUMERIC_FEATURES)
    importance = pd.Series(np.abs(sv_test[:, :k]).mean(axis=0), index=NUMERIC_FEATURES)
    importance["Satellite image (all components)"] = np.abs(sv_test[:, k:].sum(axis=1)).mean()
    importance = importance.sort_values()
    ax = importance.plot.barh(figsize=(7, 4), color="#2E8B57", title="Model 2: mean |SHAP| on log(price), test set")
    ax.set_xlabel("average impact on log(price)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "global_importance.png"), dpi=150)
    plt.show()
except Exception as e:
    print("Skipped SHAP plot:", e)

## Export for your laptop

In [ ]:
def leaf_layers(m):
    for l in m.layers:
        if isinstance(l, keras.Model):
            yield from leaf_layers(l)
        else:
            yield l

weights = {}
for l in leaf_layers(model):
    for i, w in enumerate(l.get_weights()):
        weights[f"{l.name}::{i}"] = w
np.savez(os.path.join(OUT_DIR, "weights_by_layer.npz"), **weights)

# Reference predictions computed the same way the Flask backend does (PIL resize),
# so the import script can prove the weights arrived intact.
def pil_image(path):
    img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.asarray(img, dtype="float32")
    return arr[..., ::-1] - np.array([103.939, 116.779, 123.68], dtype="float32")

ref_idx = test_idx[:16]
ref_img = np.stack([pil_image(p) for p in paths_all[ref_idx]])
ref_log = model.predict({"image_input": ref_img, "tabular_input": X_all[ref_idx]}, verbose=0).reshape(-1)
pd.DataFrame({"row": ref_idx, "image_file": df["image_file"].to_numpy()[ref_idx], "pred_log": ref_log}).to_csv(
    os.path.join(OUT_DIR, "reference_predictions.csv"), index=False)

meta = {
    "model_kind": "multimodal",
    "numeric_features": NUMERIC_FEATURES,
    "feature_median": [float(medians[c]) for c in NUMERIC_FEATURES],
    "feature_mean": [float(v) for v in mean],
    "feature_scale": [float(v) for v in scale],
    "image_size": IMAGE_SIZE,
    "target": "log1p(latestPrice)",
    "price_scale": price_scale,
    "split": "train_test_split(test_size=0.2, random_state=42) on rows with images",
    "trained_with": {"tensorflow": tf.__version__, "keras": keras.__version__},
    "quick_test": QUICK_TEST,
    "test_results": results.to_dict(orient="records"),
    "recommended_model": recommended_model,
    "boosting": {
        "params": GB_PARAMS,
        "price_scale": gb_price_scale,
        "image_features": IMAGE_FEATURES,
        "explained_variance": explained,
        "feature_extractor": "ResNet50 ImageNet, include_top=False, pooling=avg; tf.image.resize antialias; caffe preprocessing",
    },
}

np.savez(os.path.join(OUT_DIR, "hybrid_features.npz"),
         image_file=df["image_file"].to_numpy().astype(str), img_pca=img_pca, has_features=has_features,
         pca_mean=pca_mean.astype("float32"), pca_components=pca_components)
pd.DataFrame({"image_file": df["image_file"].to_numpy()[test_idx], "actual": y_test,
              "neural_hybrid": nn_test_pred, "boosting_hybrid": gb_test_pred, "numbers_only": num_test_pred}).to_csv(
    os.path.join(OUT_DIR, "test_predictions.csv"), index=False)
with open(os.path.join(OUT_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2, default=float)

zip_path = os.path.join(OUT_DIR, "aeroprice_trained_model.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["weights_by_layer.npz", "model_meta.json", "reference_predictions.csv", "training_history.csv",
                 "hybrid_features.npz", "test_predictions.csv", "global_importance.png"]:
        if os.path.exists(os.path.join(OUT_DIR, name)):
            z.write(os.path.join(OUT_DIR, name), name)
print(f"Done. Download {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB) from the Output panel.")